In [1]:
import pandas as pd
import numpy as np

In [2]:
# Load raw data
df = pd.read_csv('../data/raw_data/bank-additional-full.csv', sep=';')
print(f"Shape: {df.shape}")

Shape: (41188, 21)


## 1. Remove Duplicates
Removing 12 duplicate rows identified in exploration.

In [3]:
# Remove duplicate rows
print(f"Rows before: {df.shape[0]}")
df= df.drop_duplicates()
print(f"Rows after dropping duplicates: {df.shape[0]}")

Rows before: 41188
Rows after dropping duplicates: 41176


## 2. Remove calls with duration = 0
Removing the 4 calls with duration = 0

In [4]:
# Remove calls with duration = 0 (no contact)
print(f"Rows before: {df.shape[0]}")
df=df[df['duration']>0]
print(f"Rows after: {df.shape[0]}")

Rows before: 41176
Rows after: 41172


## 3. Merging 'illiterate' into 'basic.4y'
Only 18 rows where education= illiterate.

But every row is important: instead of deleting records with rare labels, I merged them to keep all other useful information in the others columns

I merged this category into the most conceptually similar ('basic.4y')


In [5]:
# Merging 18 'illiterate' into 'basic.4y'
df['education']= df['education'].replace('illiterate', 'basic.4y')
# Verify
print(df['education'].value_counts())

education
university.degree      12163
high.school             9510
basic.9y                6044
professional.course     5240
basic.4y                4194
basic.6y                2291
unknown                 1730
Name: count, dtype: int64


## 4. default column
Only 3 "yes" values, not statistically meaningful.

Collapsed into binary ("no" vs "unknown") by imputing the 3 irrelevant "yes" into the biggest class ("no")

I didn't drop these rows to preserve data integrity


In [6]:
# Impute 'yes' into 'no'
df['default']=df['default'].replace('yes', 'no')
# Verify
print(df['default'].value_counts())

default
no         32576
unknown     8596
Name: count, dtype: int64


## 5. pdays
`pdays` stores the number of days since the client was last contacted 
in a previous campaign. When the client was never contacted before, 
the value is 999, not a real number.

Keeping 999 as-is would confuse the model, which might interpret it 
as "contacted 999 days ago" instead of "never contacted".

**Solution:** I created a new binary column `previously_contacted`:
- `previously_contacted = 1` → client was contacted before
- `previously_contacted = 0` → client was never contacted before

Then I dropped the original 'pdays' column because keeping it would have required discarding the '999' rows. Since these non-contacts represent the majority of the dataset, I opted for a binary indicator to prevent significant data loss (irrelevant column without 39.657 non-contacts values) and avoid feature redundancy between these 2 columns.

In [7]:
# Create binary column 'previously_contacted'
df['previously_contacted']= (df['pdays']!= 999).astype(int)
# Drop 'pdays' column
df=df.drop('pdays', axis= 1)
# Verify
print(df['previously_contacted'].value_counts())
print("-" * 40)
print(df.columns)

previously_contacted
0    39657
1     1515
Name: count, dtype: int64
----------------------------------------
Index(['age', 'job', 'marital', 'education', 'default', 'housing', 'loan',
       'contact', 'month', 'day_of_week', 'duration', 'campaign', 'previous',
       'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx',
       'euribor3m', 'nr.employed', 'y', 'previously_contacted'],
      dtype='str')


## 6. Handling the 'unknown' values
'Easy' decisions:
- **"job", "marital"** -> I will use the **mode** to impute the very few "unknown" values.
- **"default"** -> a lot of "unknown" values, I will keep them as a **separate category**

### - housing and loan case (unknown values)
990 'unknown' values for each of these columns.

It is unclear whether 'unknown' reflects:
- a question not asked during the call or something not found (operational missing data)
- a deliberate refusal to answer (behavioral signal)

According to the dataset documentation, 'unknown' represents missing values; it is not specified whether these represent a refusal to answer or not.

**Decision**: impute with **mode** (2.4% of data — low risk).

If 'unknown' proves predictive in the future EDA, this decision will be revisited.


In [8]:
# Calculate job, marital, housing and loan mode
job_mode=df['job'].mode()[0]
marital_mode=df['marital'].mode()[0]
housing_mode=df['housing'].mode()[0]
loan_mode=df['loan'].mode()[0]

print(f"Job mode: {job_mode}")
print(f"\nMarital mode: {marital_mode}")
print(f"\nHousing mode: {housing_mode}")
print(f"\nLoan mode: {loan_mode}")


Job mode: admin.

Marital mode: married

Housing mode: yes

Loan mode: no


In [9]:
# Mode imputation
modes= {
    'job': job_mode,
    'marital': marital_mode,
    'housing': housing_mode,
    'loan': loan_mode
}

for col, mode in modes.items():
    df[col]=df[col].replace('unknown', mode)
    # verify
    print(f"\n{df[col].value_counts()}")


job
admin.           10748
blue-collar       9251
technician        6739
services          3967
management        2923
retired           1718
entrepreneur      1456
self-employed     1421
housemaid         1060
unemployed        1014
student            875
Name: count, dtype: int64

marital
married     24998
single      11564
divorced     4610
Name: count, dtype: int64

housing
yes    22558
no     18614
Name: count, dtype: int64

loan
no     34924
yes     6248
Name: count, dtype: int64


### - education case (unknown values)
1,730 unknown values (4.2% of dataset).

Imputing with mode (university.degree) would introduce a bias.

**Decision**: keep 'unknown' as **separate category** - large enough to be meaningful.

## 7. Save clean dataset

In [10]:
df.to_csv('../data/processed_data/bank_clean.csv', index= False)
# Final shape
print(f"Clean dataset shape: {df.shape}")

Clean dataset shape: (41172, 21)
